# Token Rate Limiting Lab

![flow](./img/token-rate-limiting.gif)

Playground to try the [token rate limiting policy](https://learn.microsoft.com/azure/api-management/azure-openai-token-limit-policy) to one or more Azure AI Foundry endpoints.

The azure-openai-token-limit policy prevents Azure OpenAI Service API usage spikes on a per key basis by limiting consumption of language model tokens to a specified number per minute. When the token usage is exceeded, the caller receives a 429 Too Many Requests response status code.

### ⚙️ Initialization

1. Get Az Cli authentication information
1. Get terraform outputs (prerequisite: run the [setup.ipynb](./setup.ipynb))

In [ ]:
import sys
sys.path.insert(1, './shared')  # add the shared directory to the Python path
import utils

terraform_dir = 'terraform'

output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

output = utils.run(f"terraform -chdir={terraform_dir} output -json", "Retrieved Terraform outputs", "Failed to retrieve Terraform outputs")

if output.success and output.json_data:
    apim_gateway_url = output.json_data['apim_gateway_url']['value']
    inference_api_path = output.json_data['token_rate_limiting']['value']['inference_api_path']
    product_api_key = output.json_data['token_rate_limiting']['value']['product_api_key']
    inference_api_version = output.json_data['token_rate_limiting']['value']['inference_api_version']
    deployment_model_name = output.json_data['token_rate_limiting']['value']['deployment_model_name']

    utils.print_info(f"APIM Gateway URL: {apim_gateway_url}")
    utils.print_info(f"Inference API path: {inference_api_path}")
    utils.print_info(f"Product key: {product_api_key}")
    utils.print_info(f"Inference API version: {inference_api_version}")
    utils.print_info(f"Deployment model name: {deployment_model_name}")

<a id='requests'></a>
### 🧪 Test the API using a direct HTTP call

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

In [ ]:
import requests, json

url = f"{apim_gateway_url}/{inference_api_path}/openai/deployments/{deployment_model_name}/chat/completions?api-version={inference_api_version}"
api_runs = []
for i in range(30):
    messages={"messages":[
        {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
        {"role": "user", "content": "Can you tell me the time, please?"}
    ]}
    response = requests.post(url, headers = {'api-key':product_api_key}, json = messages)
    if (response.status_code == 200):
        print("▶️ Run: ", i+1, "status code: ", response.status_code, "✅")
        data = json.loads(response.text)
        total_tokens = data.get("usage").get("total_tokens")
        print("💬 ", data.get("choices")[0].get("message").get("content"))
    else:
        print("▶️ Run: ", i+1, "status code: ", response.status_code, "⛔")
        print(response.text)
        total_tokens = 0
    api_runs.append((total_tokens, response.status_code))


<a id='plot'></a>
### 🔍 Analyze Token Rate limiting results


In [ ]:
# plot the results
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.figsize'] = [15, 7]
df = pd.DataFrame(api_runs, columns=['Tokens', 'Status Code'])
df['Run'] = range(1, len(df) + 1)
colors = ['red' if str(code).startswith('5') else 'yellow' if str(code).startswith('4') else 'lightblue' for code in df['Status Code']]
ax = df.plot(kind='bar', x='Run', y='Tokens', color=colors, legend=False)
plt.title('Rate Limiting results')
plt.xlabel('Runs')
plt.ylabel('Tokens')
plt.xticks(df['Run'], rotation=0)
for i, val in enumerate(df['Status Code']):
    ax.text(i, 20, '' if int(val) == 200 else '429', ha='center', va='bottom')
for i, val in enumerate(df['Tokens']):
    ax.text(i, df['Tokens'][i] + 5, '' if int(val) == 0 else val, ha='center', va='bottom')
accumulated_tokens = df['Tokens'].cumsum()
ax.plot(df['Run']-1, accumulated_tokens, color='green', label='Accumulated Tokens')
for i, val in enumerate(accumulated_tokens):
    ax.text(i, val + 6, str(int(val)), ha='center', va='bottom', label='Accumulated Tokens')
plt.show()

<a id='sdk'></a>
### 🧪 Test the API using the Azure OpenAI Python SDK

We want confirm with this test that the SDK is performing retries automatically.

In [ ]:
from openai import AzureOpenAI
for i in range(20):
    print("▶️ Run: ", i+1)

    client = AzureOpenAI(
        azure_endpoint=f"{apim_gateway_url}/{inference_api_path}",
        api_key=product_api_key,
        api_version=inference_api_version
    )
    response = client.chat.completions.create(model=deployment_model_name, messages=[
                    {"role": "system", "content": "You are a sarcastic, unhelpful assistant."},
                    {"role": "user", "content": "Can you tell me the time, please?"}
    ])
    print("💬 ",response.choices[0].message.content)


<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.